# The Medallion Architecture: Bronze, Silver, Gold

Data warehouses have existed for decades. The medallion architecture applies the same layered thinking to open file formats — Parquet and Iceberg instead of proprietary database tables. This notebook explains the pattern, contrasts it with classical data warehouse design, and shows you how to set up Apache Spark with Iceberg to work on each layer.

In [ ]:
%load_ext autoreload
%autoreload 2

## The Three Layers

The medallion architecture divides a data platform into three layers, each with a clear contract about what it guarantees.

- **Bronze**: raw data as-is, append-only, immutable. This is the source of truth. It is cheap to store (object store) and never modified. In the IoT context: raw API output from Cumulocity, unmodified — exactly what the API returned, including any quirks or duplicates.
- **Silver**: validated, deduplicated, conformed schema. Still event-grained (one row per event). Bad records have been filtered out, types have been enforced, and timestamps have been normalized. In the IoT context: clean events in Iceberg, device metadata in structured form.
- **Gold**: aggregated, denormalized, business-ready. Optimized for consumption by analytics tools, ML pipelines, and dashboards. Not event-grained — one row per time window, device group, or KPI. This is what we build in this module.

```
Raw IoT events (JSON)
       │
       ▼
  [Bronze Layer]  ← raw, immutable, append-only
  Parquet/Iceberg
       │
       ▼
  [Silver Layer]  ← validated, deduplicated, typed
  Iceberg tables
       │
       ▼
  [Gold Layer]    ← aggregated, business-ready
  Iceberg tables (written by Spark)
```

## Lakehouse vs. Classical Data Warehouse

| Dimension | Classical DWH | Lakehouse / Medallion |
|---|---|---|
| Storage | Proprietary DB engine | Open files (Parquet + Iceberg) |
| Schema | Schema-on-write (load-time validation) | Schema-on-read for bronze; schema-on-write for silver+ |
| Transform | ETL (transform before load) | ELT (load raw first, transform in place) |
| Scalability | Scale up (bigger server) | Scale out (more nodes) |
| Historical data | Expensive to retain | Cheap (object store) |
| Query engines | Single vendor lock-in | Any engine (Spark, Trino, Daft, DuckDB) |
| ACID support | Native | Via Iceberg (snapshots, optimistic locking) |

The gold layer maps directly to a data mart or star schema in classical DWH terminology. The difference is the storage format and the engine: instead of loading into a proprietary system, you write Iceberg files that any SQL engine can query.

## The Cumulocity Context

Cumulocity IoT acts as the bronze and partial silver layer in this training scenario:

- Events arrive already validated and deduplicated by the Cumulocity platform — it acts as the ingest and deduplication layer.
- The Cumulocity API provides structured JSON: typed events, device metadata, alarms — not raw sensor bytes.
- We start at silver: clean events already extracted to Iceberg, device hierarchy available in `cmdata.jsonl`.
- **Our work in this module**: build the gold layer.

This is a common pattern in enterprise data engineering: a SaaS platform or operational system acts as your bronze+silver supplier. Your data engineering work is the gold layer — turning operational data into analytical value.

> **Practical note**: In a real Cumulocity integration you would use the Cumulocity Streaming Analytics or the Data Hub product to export events. For this training course, the export has already been done and the data lives in `data/input/`.

## Setting Up PySpark with Iceberg

Before we write code, it helps to understand what we are configuring:

- **PySpark 4.0** requires Python 3.9+. The Iceberg Spark runtime JAR is downloaded automatically via `spark.jars.packages` on first SparkSession creation (requires internet access; approximately 20 MB).
- The **runtime JAR** (`iceberg-spark-runtime-4.0_2.13`) bridges PySpark's DataFrame API and SQL engine with Iceberg's Java metadata library.
- We configure two things:
  1. **SQL extensions** (`IcebergSparkSessionExtensions`): enables `MERGE INTO`, `CALL` system procedures, and time-travel syntax.
  2. **A Hadoop-backed catalog named `local`**: Spark's catalog for gold tables. No external service required — metadata lives on the local filesystem alongside the data files.
- **Why `local`?** Gold tables are written by Spark to local disk in this training environment. In production this catalog would point to S3, GCS, or ADLS, and the catalog backend would be Glue, Nessie, or a REST catalog.

The `get_spark()` helper in `helpers.py` handles all of this configuration. You pass it an `app_name` and the path to the gold warehouse, and it returns a fully configured `SparkSession`.

In [ ]:
import sys
import json
import shutil
from pathlib import Path
import pyarrow as pa
import pandas as pd
from pyiceberg.catalog.sql import SqlCatalog
import daft

sys.path.insert(0, str(Path.cwd()))
print(f"Working directory: {Path.cwd()}")

In [ ]:
# Clean up from previous runs
for d in ["../data/warehouse_silver", "../data/warehouse_gold"]:
    shutil.rmtree(d, ignore_errors=True)

# Create silver catalog
catalog = SqlCatalog("silver", **{
    "uri": "sqlite:///../data/warehouse_silver/silver_catalog.db",
    "warehouse": "file://" + str(Path("../data/warehouse_silver").resolve()),
})
catalog.create_namespace_if_not_exists("silver")

# Load events.jsonl into a PyArrow table
events = []
with open("../data/input/events.jsonl") as f:
    for line in f:
        events.append(json.loads(line))

df_events = pd.DataFrame(events)
arrow_table = pa.Table.from_pandas(df_events)

# Write to silver Iceberg table
if catalog.table_exists("silver.events"):
    catalog.drop_table("silver.events")
silver_events = catalog.create_table("silver.events", schema=arrow_table.schema)
silver_events.append(arrow_table)

print(f"Silver events table: {len(events):,} rows")
print(f"Location: {silver_events.location()}")
print(f"Schema fields: {[f.name for f in silver_events.schema().fields]}")

In [ ]:
from helpers import get_spark

spark = get_spark("MedallionArchitecture", gold_warehouse="../data/warehouse_gold")
print(f"Spark version: {spark.version}")
print(f"Iceberg extensions loaded: {spark.conf.get('spark.sql.extensions')}")

## Reading the Silver Layer from Spark

PyIceberg uses a SqlCatalog (SQLite-backed). Spark uses a Hadoop-backed catalog. These are different catalog backends — they cannot share the same SQLite database, and there is no cross-catalog federation in the local training setup.

But Iceberg is fundamentally a **file format**: the table data and metadata are just files on disk. Every Iceberg table has a `metadata/` directory with JSON files describing its snapshots, schemas, and partition specs, and a `data/` directory with Parquet files.

Spark can read any Iceberg table **directly by its filesystem path**, bypassing the catalog entirely. This is the cross-catalog pattern we use throughout this module:

```python
silver_df = spark.read.format("iceberg").load(silver_path)
```

In production on AWS, the silver layer would live in S3 behind a **shared catalog** (Glue or a REST catalog). Both PyIceberg and Spark would register and discover tables through the same catalog, so you would use table names (`silver.events`) instead of paths. The path-based approach is a training convenience, not a production recommendation.

In [ ]:
# PyIceberg → Spark via Arrow: no shared catalog required
silver_df = spark.createDataFrame(silver_events.scan().to_arrow().to_pandas())
print(f"Rows: {silver_df.count():,}")
print(f"Partitions: {silver_df.rdd.getNumPartitions()}")
silver_df.printSchema()

In [ ]:
# Event type distribution
print("Event types:")
silver_df.groupBy("type").count().orderBy("count", ascending=False).show(10)

# Active devices per day
from pyspark.sql import functions as F

silver_df.withColumn("date", F.to_date("time")).groupBy("date").agg(
    F.countDistinct("source").alias("active_devices"),
    F.count("*").alias("events")
).orderBy("date").show(10)

## Writing to the Gold Layer

Gold tables are written by Spark to its own Hadoop catalog (`local`). A few things to understand:

- Table names follow the pattern `local.<namespace>.<table>` — both in SQL statements (`SELECT * FROM local.gold.my_table`) and in the DataFrame writer API (`df.writeTo("local.gold.my_table")`).
- Spark creates Iceberg metadata (manifest lists, manifests, Parquet data files) just like PyIceberg does. The on-disk layout is identical.
- The gold table can be read back with Spark SQL, the PySpark DataFrame API, or any Iceberg-compatible reader (Daft, Trino, DuckDB with the Iceberg extension, etc.).
- `createOrReplace()` is the DataFrame writer equivalent of `CREATE OR REPLACE TABLE`. It is idempotent — safe to re-run in a notebook.

In [ ]:
from pyspark.sql import functions as F

spark.sql("CREATE NAMESPACE IF NOT EXISTS local.gold")

# Gold: daily event counts by type (simplest possible gold table)
daily_by_type = silver_df.withColumn(
    "event_date", F.to_date("time")
).groupBy("event_date", "type").agg(
    F.count("*").alias("event_count")
).orderBy("event_date", "type")

daily_by_type.writeTo("local.gold.daily_events_by_type").createOrReplace()

print("Gold table written.")
spark.sql("SELECT * FROM local.gold.daily_events_by_type ORDER BY event_date DESC LIMIT 10").show()

In [ ]:
# Snapshot history
print("Snapshot history:")
spark.sql("SELECT * FROM local.gold.daily_events_by_type.history").show()

# Files written
print("\nData files:")
spark.sql("SELECT file_path, record_count, file_size_in_bytes FROM local.gold.daily_events_by_type.files").show(truncate=False)

## The Lakehouse in Action

Let us take stock of what just happened across the two processes:

- **Silver events** were stored by a PyIceberg process (Python, SqlCatalog, SQLite backend). Spark read them without any catalog coordination — purely by pointing at the table's filesystem path. No shared service, no schema registration.
- **Gold** was written by Spark to its own Hadoop-backed catalog. The output is plain Iceberg files: Parquet data files and JSON metadata files. Any Iceberg-compatible reader can consume them.
- Both layers are **open formats**: no proprietary serialization, no engine lock-in. You could shut down Spark, install Trino, and query the same gold files immediately.

In production, the setup is the same in structure but different in infrastructure:
- Object store (S3/GCS/ADLS) replaces the local filesystem.
- A shared catalog (Glue, Nessie, or a REST catalog) replaces the per-engine catalogs, so table names work across all engines.
- The engines remain interchangeable: you can migrate from Spark to Trino or Daft without touching the data files.

In [ ]:
import daft
import glob

# Read the Spark-written gold table with PyIceberg/Daft (using the hadoop catalog)
# We need to point PyIceberg to the same warehouse path
gold_path = str(Path("../data/warehouse_gold/gold/daily_events_by_type").resolve())
# Daft can read any Iceberg table by path using its metadata path
metadata_files = sorted(glob.glob(f"../data/warehouse_gold/gold/daily_events_by_type/metadata/*.json"))
print(f"Gold table metadata files: {len(metadata_files)}")
for m in metadata_files[-3:]:
    print(f"  {Path(m).name}")
print("\nGold table is plain Iceberg files — readable by any Iceberg-compatible engine.")

## Review Questions

1. What is the key difference between the silver and gold layers?
2. Why does the lakehouse use ELT instead of ETL?
3. What makes Iceberg table data readable by multiple engines without a shared catalog?
4. You have raw Cumulocity event JSON files in S3. Sketch the medallion pipeline: what happens at each layer?
5. A colleague suggests storing the gold table in PostgreSQL. What does Iceberg offer that PostgreSQL does not in this context?

## Challenges

- Query the silver events table with Daft instead of Spark. What SQL syntax differences do you notice?
- Add a second gold table: `device_daily_active` — one row per `(date, device_id)` with `event_count`. Write it with Spark and inspect its snapshot history.
- Time travel on the gold table: append more data, then query the previous snapshot using `VERSION AS OF`.
- Sketch the medallion design for a factory floor monitoring use case: what events would be bronze, what quality checks define silver, and what KPIs define gold?

## Summary

- The medallion architecture organizes data into **bronze** (raw), **silver** (clean), and **gold** (aggregated) layers, each with clear contracts about what guarantees the data provides.
- The **lakehouse pattern** uses open file formats (Parquet + Iceberg) instead of a proprietary database, enabling any query engine to access any layer.
- **PySpark 4.0** with the Iceberg Spark runtime reads tables written by PyIceberg (and vice versa) without a shared catalog — just by pointing at the table's filesystem location.
- The **gold layer** maps to what classical data warehouses call a data mart. The difference is the storage format and the ability to use multiple query engines without data movement.
- **Next**: building the gold layer in depth — time-window aggregations, asset hierarchy rollups, and slowly changing dimension (SCD) patterns.